# Exercise 05 — Advanced Tools

Topics covered:
1. **Batch Tool** — trick Claude into running multiple tools in parallel
2. **Forced Structured Extraction** — use `tool_choice` to always get JSON output
3. **Web Search Tool** — built-in tool that searches and cites sources automatically

In [ ]:
import json
from dotenv import load_dotenv
import anthropic
from anthropic.types import ToolParam

load_dotenv(dotenv_path="../.env")
client = anthropic.Anthropic()
MODEL = "claude-sonnet-4-5"

print("Ready!")

## Part 1 — The Batch Tool

**Problem:** When you have multiple independent tools, Claude often calls them one at a time (sequential), even though they could run in parallel.

**Solution:** Create a "batch" tool that wraps multiple tool calls into a single request. Claude calls the batch tool once, and your code runs all sub-tools and returns all results at once.

In [ ]:
# Individual tool functions
def get_weather(city: str) -> dict:
    """Mock weather API call."""
    weather_data = {
        "london": {"temp_c": 15, "condition": "cloudy", "humidity": 80},
        "paris":  {"temp_c": 20, "condition": "sunny",  "humidity": 55},
        "tokyo":  {"temp_c": 28, "condition": "humid",  "humidity": 90},
        "sydney": {"temp_c": 22, "condition": "clear",  "humidity": 65},
    }
    key = city.lower()
    if key not in weather_data:
        raise ValueError(f"No weather data for city: {city!r}. Available: {list(weather_data.keys())}")
    return {"city": city, **weather_data[key]}


def get_population(city: str) -> dict:
    """Mock population data."""
    populations = {
        "london": 9_000_000,
        "paris":  2_200_000,
        "tokyo": 14_000_000,
        "sydney": 5_300_000,
    }
    key = city.lower()
    if key not in populations:
        raise ValueError(f"No population data for city: {city!r}")
    return {"city": city, "population": populations[key]}


# Individual schemas
get_weather_schema = ToolParam(
    name="get_weather",
    description="Returns current weather data for a city.",
    input_schema={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name."}
        },
        "required": ["city"]
    }
)

get_population_schema = ToolParam(
    name="get_population",
    description="Returns the population of a city.",
    input_schema={
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "City name."}
        },
        "required": ["city"]
    }
)

print("Individual tools defined.")

In [ ]:
# The batch tool schema
batch_tool_schema = ToolParam(
    name="batch",
    description=(
        "Run multiple tools in a single call. Use this when you need to call several "
        "independent tools whose results don't depend on each other — it's faster than "
        "calling them one at a time. Each invocation specifies a tool name and its arguments."
    ),
    input_schema={
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "List of tool calls to execute in parallel.",
                "items": {
                    "type": "object",
                    "properties": {
                        "tool_name": {"type": "string", "description": "Name of the tool to call."},
                        "arguments": {"type": "string", "description": "JSON-encoded arguments for the tool."}
                    },
                    "required": ["tool_name", "arguments"]
                }
            }
        },
        "required": ["invocations"]
    }
)


def run_single_tool(tool_name: str, tool_input: dict) -> tuple[str, bool]:
    """Dispatch a single tool call. Returns (result_json, is_error)."""
    try:
        if tool_name == "get_weather":
            result = get_weather(**tool_input)
        elif tool_name == "get_population":
            result = get_population(**tool_input)
        else:
            raise ValueError(f"Unknown tool: {tool_name!r}")
        return json.dumps(result), False
    except Exception as e:
        return str(e), True


def run_batch(invocations: list[dict]) -> list[dict]:
    """Execute all sub-tool calls and return their results."""
    batch_output = []
    for inv in invocations:
        tool_name = inv["tool_name"]
        arguments = json.loads(inv["arguments"])
        result, is_error = run_single_tool(tool_name, arguments)
        batch_output.append({
            "tool_name": tool_name,
            "result":    result,
            "is_error":  is_error
        })
    return batch_output


def run_tool_with_batch(tool_name: str, tool_input: dict) -> tuple[str, bool]:
    """Extended dispatcher that also handles the batch tool."""
    if tool_name == "batch":
        result = run_batch(tool_input["invocations"])
        return json.dumps(result), False
    return run_single_tool(tool_name, tool_input)


ALL_TOOLS_WITH_BATCH = [get_weather_schema, get_population_schema, batch_tool_schema]

print("Batch tool ready.")

In [ ]:
def run_conversation(user_input: str, tools: list, verbose: bool = True) -> str:
    """Generic conversation loop for this notebook."""
    messages = [{"role": "user", "content": user_input}]
    while True:
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            tools=tools,
            messages=messages
        )
        messages.append({"role": "assistant", "content": response.content})

        tool_results = []
        for block in response.content:
            if hasattr(block, "text") and block.text and verbose:
                print(f"  [Claude] {block.text}")
            elif block.type == "tool_use":
                if verbose:
                    print(f"  [Tool]   {block.name}({json.dumps(block.input)[:80]})")
                result_content, is_error = run_tool_with_batch(block.name, block.input)
                if verbose:
                    print(f"  [Result] {result_content[:100]}")
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result_content,
                    "is_error": is_error,
                })

        if response.stop_reason != "tool_use":
            break
        messages.append({"role": "user", "content": tool_results})

    return " ".join(
        b.text for b in response.content if hasattr(b, "text")
    )


# Test: multi-city query — Claude should use the batch tool
print("USER: Compare the weather and population of London, Paris, and Tokyo.\n")
result = run_conversation(
    "Compare the current weather and population of London, Paris, and Tokyo.",
    tools=ALL_TOOLS_WITH_BATCH
)
print(f"\nFINAL REPLY:\n{result}")

## Part 2 — Forced Structured Extraction with `tool_choice`

Instead of pre-fill + stop sequences, you can use a tool schema to **force** Claude to always output a specific JSON structure. This is more reliable for complex schemas.

**How it works:**
1. Define a tool whose input_schema matches your desired output JSON shape
2. Set `tool_choice={"type": "tool", "name": "your_tool"}` to **force** Claude to call it
3. Extract `response.content[0].input` — it's already a Python dict, no parsing needed

In [ ]:
# Define the schema for the data we want to extract
extract_invoice_schema = ToolParam(
    name="extract_invoice",
    description="Extract structured invoice data from text.",
    input_schema={
        "type": "object",
        "properties": {
            "invoice_number": {"type": "string",  "description": "Invoice number or ID."},
            "vendor_name":    {"type": "string",  "description": "Name of the vendor."},
            "total_amount":   {"type": "number",  "description": "Total amount as a float."},
            "currency":       {"type": "string",  "description": "3-letter currency code."},
            "due_date":       {"type": "string",  "description": "Due date in YYYY-MM-DD format."},
            "line_items": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": {
                        "description": {"type": "string"},
                        "quantity":    {"type": "number"},
                        "unit_price":  {"type": "number"},
                        "total":       {"type": "number"}
                    }
                },
                "description": "List of line items on the invoice."
            }
        },
        "required": ["invoice_number", "vendor_name", "total_amount", "currency"]
    }
)


invoice_text = """
INVOICE #INV-2024-0892
From: Acme Software Solutions Ltd
Date: March 15, 2024
Payment due: April 14, 2024

Items:
- Annual software license (5 seats)  x1  @ £1,200.00  = £1,200.00
- Priority support package           x1  @ £450.00    = £450.00
- Training session (4hrs)            x2  @ £175.00    = £350.00

Total due: £2,000.00 GBP
"""

# Force Claude to always call the extract_invoice tool
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=[extract_invoice_schema],
    tool_choice={"type": "tool", "name": "extract_invoice"},
    messages=[{
        "role": "user",
        "content": f"Extract the invoice data from this text:\n{invoice_text}"
    }]
)

# Access the structured data — already a Python dict, no json.loads() needed
invoice_data = response.content[0].input
print("Extracted invoice data:")
print(json.dumps(invoice_data, indent=2))

### Exercise 2a — Extract Resume Data

Define a tool schema to extract structured resume data from the text below. Fields to extract: `name`, `email`, `skills` (list), `years_experience`, `current_role`.

In [ ]:
resume_text = """
Jane Doe | jane.doe@email.com | LinkedIn: linkedin.com/in/janedoe

EXPERIENCE
Senior Data Scientist @ TechCorp (2020–present)
Data Analyst @ StartupX (2018–2020)
Junior Developer @ WebAgency (2016–2018)

SKILLS
Python, SQL, TensorFlow, PyTorch, Spark, Tableau, Docker, AWS
"""

# TODO: Define extract_resume_schema with the fields described above
extract_resume_schema = ToolParam(
    name="extract_resume",
    description="Extract structured resume data from text.",
    input_schema={
        "type": "object",
        "properties": {
            # TODO: add field definitions here
        },
        "required": []
    }
)

response = client.messages.create(
    model=MODEL,
    max_tokens=512,
    tools=[extract_resume_schema],
    tool_choice={"type": "tool", "name": "extract_resume"},
    messages=[{"role": "user", "content": f"Extract resume data:\n{resume_text}"}]
)

resume_data = response.content[0].input
print(json.dumps(resume_data, indent=2))

## Part 3 — Built-in Web Search Tool

The web search tool is completely built into Claude — you only provide the schema, Claude handles the actual searching AND returns citations showing where each statement came from.

You can optionally restrict searches to specific domains.

In [ ]:
# Basic web search — unrestricted
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 3,
}

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=[web_search_schema],
    messages=[{"role": "user", "content": "What is the latest version of Python and when was it released?"}]
)

# Display the response with citation info
print("Response content blocks:")
for block in response.content:
    block_type = getattr(block, "type", "unknown")
    print(f"  Type: {block_type}")
    if hasattr(block, "text"):
        print(f"  Text: {block.text[:200]}...")
    elif hasattr(block, "query"):
        print(f"  Search query: {block.query}")
    print()

In [ ]:
# Restricted web search — only search CDC for health info
restricted_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 3,
    "allowed_domains": ["cdc.gov", "who.int"],  # only trusted health sources
}

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=[restricted_search_schema],
    messages=[{"role": "user", "content": "What are the CDC guidelines for daily physical activity for adults?"}]
)

# Extract just the text response
text_response = " ".join(
    block.text for block in response.content
    if hasattr(block, "text")
)
print(text_response)

### Exercise 3a — Domain-Restricted Search

Create a web search that is restricted to `python.org` and `docs.python.org`. Ask it: _"How do I use async/await in Python?"_ and display the text response.

In [ ]:
# TODO: Create a search restricted to official Python documentation
python_docs_search = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 3,
    # TODO: add allowed_domains
}

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=[python_docs_search],
    messages=[{"role": "user", "content": "How do I use async/await in Python?"}]
)

text_response = " ".join(
    block.text for block in response.content
    if hasattr(block, "text")
)
print(text_response)

## Summary

- ✅ **Batch tool** — wrap multiple tools so Claude runs them in a single round-trip
- ✅ **Forced extraction** — `tool_choice` guarantees structured JSON output with no parsing needed
- ✅ **Web search** — zero implementation required; Claude searches and cites sources automatically

**Next:** [06_rag_pipeline.ipynb](06_rag_pipeline.ipynb) — building a Retrieval Augmented Generation pipeline from scratch